# PyTorch nn.Transformer 接口讲解：Encoder 与 Decoder 的使用

本 Notebook 专注于讲解如何直接使用 PyTorch 内置的 `nn.Transformer`、`nn.TransformerEncoder`、`nn.TransformerDecoder` 三套接口，
并深入解析这些接口的**掩码（mask）设计哲学**与手写实现版本（Notebook 2）的差异与原因。

> **说明**：本 Notebook 不做完整的训练流程，仅讲解 API 用法和关键概念。

## 一、环境准备

In [29]:
import torch                      # 导入 PyTorch 主库，提供张量计算与自动求导框架
import torch.nn as nn             # 导入神经网络模块，提供 Transformer 相关封装组件
import torch.nn.functional as F   # 导入函数式 API，用于演示底层注意力接口

# 打印 PyTorch 版本，便于环境复现
print('torch version:', torch.__version__)  # 返回值：str，如 '2.12.0+cu132'

# 自动选择计算设备：优先 GPU（cuda:0），无 GPU 时回退 CPU
device = torch.device('cuda:0') if torch.cuda.is_available() else torch.device('cpu')  # 返回值：torch.device
print('device:', device)          # 输出当前使用设备，如 'cuda:0' 或 'cpu'

torch version: 2.12.0+cu132
device: cuda:0


## 二、掩码设计的核心差异：nn.Transformer vs 手写实现

在深入学习各 API 之前，先理解**最重要的概念差异**：掩码的形状与布尔语义。

---

### 2.1 手写实现中的掩码设计（Notebook 2 的方式）

在 Notebook 2 的自定义 `MultiHeadAttention` 中：

```
attn_mask 形状：(batch, 1, seq_q, seq_k)
```

- **第 0 维 `batch`**：每个样本独立控制掩码
- **第 1 维 `1`**：利用广播，所有注意力头共享同一掩码，无需重复 num_heads 次
- **第 2 维 `seq_q`**：Query 序列长度
- **第 3 维 `seq_k`**：Key 序列长度
- **布尔语义**：`True / 1` 表示**需要被遮盖**（mask=1 → 乘以 -1e9 后 softmax 趋近于 0）

设计初衷：让 `attn_mask` 可以在 `num_heads` 维度上自动广播，减少内存占用。

---

### 2.2 nn.Transformer 中的掩码设计

**底层调用链**（理解掩码差异的基础）：

```
nn.Transformer
  ├── nn.TransformerEncoder
  │     └── nn.TransformerEncoderLayer × num_encoder_layers
  │           └── nn.MultiheadAttention   # 自注意力
  │                 └── F.scaled_dot_product_attention  # 底层单头计算
  └── nn.TransformerDecoder
        └── nn.TransformerDecoderLayer × num_decoder_layers
              ├── nn.MultiheadAttention   # 自注意力
              │     └── F.scaled_dot_product_attention
              └── nn.MultiheadAttention   # 交叉注意力（Cross-Attention）
                    └── F.scaled_dot_product_attention
```

> `nn.MultiheadAttention` 负责线性投影、拆分多头、合并输出；`F.scaled_dot_product_attention` 只负责单头的 $QK^T/\sqrt{d_k}$ + softmax 计算。掩码的 bool 语义转换（`True=忽略` → `True=保留`）发生在 `nn.MultiheadAttention` 内部，对用户透明。

PyTorch 官方接口将掩码拆分为两类，形状完全不同：

> **形状字母含义**：N = batch size（批次大小）；S = Source 源序列长度（Encoder 端）；T = Target 目标序列长度（Decoder 端）；nhead = 注意力头数

| 参数名 | 形状 | 作用 |
|---|---|---|
| `tgt_mask` | **(T, T)** 或 (N·nhead, T, T) | Decoder 自注意力掩码，因果掩码防止看到未来 token |
| `src_mask` | **(S, S)** 或 (N·nhead, S, S) | Encoder 自注意力掩码，通常为 None（全可见） |
| `src_key_padding_mask` | **(N, S)** | 告知模型哪些 token 是 padding |
| `tgt_key_padding_mask` | **(N, T)** | 告知模型目标序列哪些是 padding |
| `memory_key_padding_mask` | **(N, S)** | 交叉注意力时，encoder 输出哪些位置是 padding |

> **为什么 `tgt_mask` / `src_mask` 不含 N（batch）维？**
>
> `_mask` 描述的是序列内部位置之间的可见关系（例如因果掩码：位置 t 只能看到位置 0…t），这个结构对 batch 内**所有样本完全相同**——无论哪条句子，第 t 个 token 都不能看到第 t+1 个 token。
> 因此只需一份 `(T, T)` 矩阵，框架内部广播到全部 N 个样本，无需重复 N 次。
>
> 相比之下，`_key_padding_mask` 描述的是「哪些位置是 padding」，这是**数据相关**的——batch 内每条句子长度不同，padding 位置各异，因此必须保留 N 维来区分每个样本。

**关键差异总结**：

1. **没有 batch 维**（`_mask` 系列）：`tgt_mask` 形状为 `(T, T)`，不含 batch 维度，模型内部自动广播至所有样本
2. **没有 num_heads 维**（`_key_padding_mask` 系列）：形状为 `(N, S)`，比 Notebook 2 中的 `(batch, 1, seq_q, seq_k)` 少了两个维度
3. **bool 语义与 F.sdpa 相反**（重要！）：`nn.Transformer` 中 bool 掩码 `True = 忽略该位置`，而 `F.scaled_dot_product_attention` 的 `attn_mask` 为 bool 时 `True = 保留该位置`

---

### 2.3 为什么 nn.Transformer 不需要 (bs, num_heads, q_len, k_len) 形状？

**原因一：职责分离**

`_key_padding_mask` 的语义是『这个 token 是 padding』，与注意力头数无关——所有头对 padding 的处理方式相同，因此只需 `(N, S)` 即可，框架内部会自动扩展到所有头。

**原因二：内部实现差异**

PyTorch 在内部调用 SDPA（Scaled Dot-Product Attention）之前，会将两类掩码**合并**。
以 Decoder 自注意力为例，合并过程分三步：

```
Step 1 — 因果掩码升维：
  tgt_mask (T, T)
    → unsqueeze(0).unsqueeze(0)
    → (1, 1, T, T)          # 在最前面补 batch 维和 head 维（值为 1，可广播）

Step 2 — padding 掩码升维：
  tgt_key_padding_mask (N, T)
    → unsqueeze(1).unsqueeze(2)
    → (N, 1, 1, T)          # 在 batch 后补 head 维和 query 维（值为 1，可广播）

Step 3 — 两者相加（标准 numpy 广播规则）：
  (1, 1, T, T)
+ (N, 1, 1, T)
= (N, 1, T, T)              # 此时 head 维仍为 1

Step 4 — 加到注意力分数上时再次广播：
  注意力分数形状为 (N, num_heads, T, T)
  (N, 1, T, T) 广播到 (N, num_heads, T, T)  # head 维的 1 扩展为 num_heads
```

> **结论**：`num_heads` 维不是两个掩码相加得到的，而是在最后与注意力分数相加时广播出来的。
> 两个掩码合并后的中间形状是 `(N, 1, T, T)`，仍然只有一份，所有头共享。

交叉注意力端同理：`memory_mask (T, S)` + `memory_key_padding_mask (N, S)` → `(N, 1, T, S)` → 广播到 `(N, num_heads, T, S)`

手写实现之所以使用 `(batch, 1, seq_q, seq_k)`，是因为它将两者的职责合并在一个张量里，并通过 `1` 维度让头数广播。

**原因三：API 易用性**

用户只需传入语义清晰的 `(N, S)` 掩码，无需关心内部多头计算的维度布局，降低了使用门槛。

## 三、掩码形状对比可视化

In [30]:
# ── 演示两种掩码的形状差异 ──────────────────────────────────────────────

# 基础超参数
N         = 2   # batch_size：批次大小，int
num_heads = 4   # 注意力头数，int；用于演示框架内部将掩码扩展至 (N, num_heads, T, T) 的效果
S         = 6   # 源序列长度（Source sequence length），int
T         = 4   # 目标序列长度（Target sequence length），int

# ── 方式一：Notebook 2（手写实现）的掩码形状 ──────────────────────────
# 将 padding 信息和因果信息压缩到同一个张量，dim=1 固定为 1 以广播至所有头
mask_notebook2 = torch.zeros(N, 1, T, S, dtype=torch.bool)  # 形状 (batch, 1, seq_q, seq_k)，bool 类型
mask_notebook2[0, 0, :, 4:] = True  # 第 0 个样本：Key 位置 4、5 是 padding，所有 Query 都不能关注它们
print('Notebook 2 手写掩码形状:', mask_notebook2.shape)  # -> torch.Size([2, 1, 4, 6])

# ── 方式二：nn.Transformer 的 _key_padding_mask（padding 信息单独一个张量）
key_padding_mask = torch.zeros(N, S, dtype=torch.bool)  # 形状 (batch, seq_len)，bool 类型；True=padding
key_padding_mask[0, 4:] = True  # 第 0 个样本：位置 4、5 是 padding
print('nn.Transformer padding 掩码形状:', key_padding_mask.shape)  # -> torch.Size([2, 6])

# ── 方式三：nn.Transformer 的因果掩码（仅描述序列内部可见关系，不含 batch）
# generate_square_subsequent_mask 生成上三角 -inf 掩码，防止解码器提前看到未来 token
tgt_mask = nn.Transformer.generate_square_subsequent_mask(T)  # 形状 (T, T)，float 类型
print('nn.Transformer 因果掩码形状:', tgt_mask.shape)  # -> torch.Size([4, 4])
print('因果掩码内容（-inf 表示不可见）:\n', tgt_mask)  # 上三角为 -inf，下三角（含对角线）为 0

# ── bool 语义对比说明 ──────────────────────────────────────────────────
print()
print('[Notebook 2]            mask=True  -> 该位置被遮盖（乘以 -1e9 -> softmax 后趋近 0）')
print('[nn.Transformer bool]   True       -> 忽略该位置（与 Notebook 2 语义相同）')
print('[F.scaled_dot_product_attention bool] True -> 保留该位置（与上两者相反！）')

# ── 演示框架内部如何将 (T,T) 因果掩码扩展至 (N, num_heads, T, T) ────────────
# unsqueeze(0).unsqueeze(0)：在最前面插入 batch 和 num_heads 两个维度
# expand：将 1 维广播复制为 N 和 num_heads 倍，不占用额外内存
tgt_mask_expanded = tgt_mask.unsqueeze(0).unsqueeze(0).expand(N, num_heads, T, T)  # 形状 (N, num_heads, T, T) = (2, 4, 4, 4)，dtype=float32
print('因果掩码扩展后形状（框架内部等效）:', tgt_mask_expanded.shape)  # -> torch.Size([2, 4, 4, 4])

Notebook 2 手写掩码形状: torch.Size([2, 1, 4, 6])
nn.Transformer padding 掩码形状: torch.Size([2, 6])
nn.Transformer 因果掩码形状: torch.Size([4, 4])
因果掩码内容（-inf 表示不可见）:
 tensor([[0., -inf, -inf, -inf],
        [0., 0., -inf, -inf],
        [0., 0., 0., -inf],
        [0., 0., 0., 0.]])

[Notebook 2]            mask=True  -> 该位置被遮盖（乘以 -1e9 -> softmax 后趋近 0）
[nn.Transformer bool]   True       -> 忽略该位置（与 Notebook 2 语义相同）
[F.scaled_dot_product_attention bool] True -> 保留该位置（与上两者相反！）
因果掩码扩展后形状（框架内部等效）: torch.Size([2, 4, 4, 4])


## 四、nn.TransformerEncoder 使用讲解

`nn.TransformerEncoder` 是纯编码器模块，由多个 `TransformerEncoderLayer` 堆叠组成。

- 每层结构：**多头自注意力 → Add & Norm → FFN → Add & Norm**
- 不含词嵌入和位置编码，输入须为已嵌入的浮点张量

In [31]:
# ── Step 1：构造单层 EncoderLayer ─────────────────────────────────────
encoder_layer = nn.TransformerEncoderLayer(
    d_model=256,        # 特征（嵌入）维度，int；Q/K/V 的线性层输入输出维度均为 d_model
    nhead=8,            # 多头注意力的头数，int；要求 d_model % nhead == 0
    dim_feedforward=512,# FFN 隐藏层维度，int；先升维到 512 再降回 256
    dropout=0.1,        # Dropout 概率，float；训练时随机丢弃神经元防止过拟合
    batch_first=True,   # bool；True 表示输入形状为 (N, S, E)，False 为 (S, N, E)
)  # 返回值：nn.TransformerEncoderLayer 实例

# ── Step 2：堆叠多个 EncoderLayer 构成完整 Encoder ────────────────────
encoder = nn.TransformerEncoder(
    encoder_layer,              # 单层 EncoderLayer 模板，内部会深拷贝 num_layers 份
    num_layers=6,               # 堆叠层数，int；论文原版为 6
    enable_nested_tensor=False, # bool；禁用 Nested Tensor 内部优化，避免 prototype-stage 警告
                                # 该优化在传入 src_key_padding_mask 时自动触发，但目前 API 尚不稳定
)  # 返回值：nn.TransformerEncoder 实例

print(encoder)  # 打印模型结构，展示 6 层 EncoderLayer 的嵌套关系

TransformerEncoder(
  (layers): ModuleList(
    (0-5): 6 x TransformerEncoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (linear1): Linear(in_features=256, out_features=512, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=512, out_features=256, bias=True)
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
    )
  )
)


In [32]:
# ── 构造输入数据 ─────────────────────────────────────────────────────
N_enc, S_enc, E_enc = 2, 15, 256  # N=batch大小, S=源序列长度, E=特征维度，均为 int

# 模拟已完成词嵌入 + 位置编码后的特征张量（实际使用时需自行叠加位置编码）
src = torch.rand(N_enc, S_enc, E_enc)  # 随机张量，形状 (2, 15, 256)，dtype=float32

# ── 构造 padding 掩码 ──────────────────────────────────────────────
# src_key_padding_mask：形状 (N, S)，bool 类型
# True  = 该位置是 padding，自注意力时会被忽略（不参与 softmax 归一化）
# False = 真实 token，正常参与注意力计算
src_key_padding_mask = torch.zeros(N_enc, S_enc, dtype=torch.bool)  # 初始全 False，形状 (2, 15)
src_key_padding_mask[0, 12:] = True  # 第 0 个样本：位置 12、13、14 是 padding

print('src 形状:', src.shape)                                     # -> torch.Size([2, 15, 256])
print('src_key_padding_mask 形状:', src_key_padding_mask.shape)   # -> torch.Size([2, 15])
print('src_key_padding_mask[0] (True=padding):\n', src_key_padding_mask[0])  # 前 12 个 False，后 3 个 True

src 形状: torch.Size([2, 15, 256])
src_key_padding_mask 形状: torch.Size([2, 15])
src_key_padding_mask[0] (True=padding):
 tensor([False, False, False, False, False, False, False, False, False, False,
        False, False,  True,  True,  True])


In [33]:
# ── 执行 Encoder 前向传播 ──────────────────────────────────────────────
encoder.eval()  # 切换到评估模式，关闭 Dropout，确保输出确定性

with torch.no_grad():  # 关闭梯度计算，节省显存，讲解演示时不需要反向传播
    enc_out = encoder(
        src,                                          # 已嵌入的源序列，形状 (N, S, E) = (2, 15, 256)
        src_key_padding_mask=src_key_padding_mask,    # padding 掩码，形状 (N, S) = (2, 15)，True=忽略
    )  # 返回值：Tensor，形状 (N, S, E) = (2, 15, 256)，每个位置的上下文特征

print('Encoder 输出形状:', enc_out.shape)
# -> torch.Size([2, 15, 256])
# 输出序列长度与输入相同（S=15），编码器不改变序列长度，只更新每个位置的语义表示

Encoder 输出形状: torch.Size([2, 15, 256])


## 五、nn.TransformerDecoder 使用讲解

`nn.TransformerDecoder` 是解码器模块，由多个 `TransformerDecoderLayer` 堆叠组成。

- 每层结构：**因果自注意力 → Add & Norm → 交叉注意力（与 Encoder 输出交互）→ Add & Norm → FFN → Add & Norm**
- 接受两个输入：目标序列嵌入 `tgt` 和编码器输出 `memory`
- 同样不含词嵌入

In [34]:
# ── Step 1：构造单层 DecoderLayer ─────────────────────────────────────
decoder_layer = nn.TransformerDecoderLayer(
    d_model=256,        # 特征维度，int；须与 Encoder 的 d_model 保持一致
    nhead=8,            # 多头注意力头数，int；自注意力和交叉注意力均使用此头数
    dim_feedforward=512,# FFN 隐藏层维度，int
    dropout=0.1,        # Dropout 概率，float
    batch_first=True,   # bool；True 表示输入形状为 (N, T, E)
)  # 返回值：nn.TransformerDecoderLayer 实例

# ── Step 2：堆叠多层构成完整 Decoder ──────────────────────────────────
decoder = nn.TransformerDecoder(
    decoder_layer,      # 单层 DecoderLayer 模板
    num_layers=6,       # 堆叠层数，int
)  # 返回值：nn.TransformerDecoder 实例

print(decoder)  # 打印模型结构，每层包含 self_attn、multihead_attn（交叉注意力）、linear1/2

TransformerDecoder(
  (layers): ModuleList(
    (0-5): 6 x TransformerDecoderLayer(
      (self_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (multihead_attn): MultiheadAttention(
        (out_proj): NonDynamicallyQuantizableLinear(in_features=256, out_features=256, bias=True)
      )
      (linear1): Linear(in_features=256, out_features=512, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (linear2): Linear(in_features=512, out_features=256, bias=True)
      (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm2): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (norm3): LayerNorm((256,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout1): Dropout(p=0.1, inplace=False)
      (dropout2): Dropout(p=0.1, inplace=False)
      (dropout3): Dropout(p=0.1, inplace=False)
    )
  )
)


In [35]:
# ── 构造 Decoder 输入数据 ──────────────────────────────────────────────
N_dec, T_dec, S_dec, E_dec = 2, 10, 15, 256  # N=batch, T=目标序列长, S=源序列长, E=特征维度

# memory：Encoder 的输出，也即 Decoder 交叉注意力的 Key 和 Value 来源
# 在完整流程中传入 enc_out；此处为演示单独使用 Decoder，用随机张量代替
memory = torch.rand(N_dec, S_dec, E_dec)  # 形状 (2, 15, 256)，dtype=float32，模拟 Encoder 输出

# tgt：目标序列的嵌入表示（训练时为 teacher-forcing 序列，推理时逐步生成）
tgt = torch.rand(N_dec, T_dec, E_dec)  # 形状 (2, 10, 256)，dtype=float32

# ── 构造因果掩码（Causal Mask）────────────────────────────────────────
# generate_square_subsequent_mask(T) 生成形状 (T, T) 的上三角 -inf 掩码（float）
# 语义：第 t 行表示「第 t 个 Query 只能看到位置 0..t 的 Key」
# 作用：防止解码器在预测第 t 个 token 时偷看第 t+1...T-1 个 token（自回归约束）
tgt_mask = nn.Transformer.generate_square_subsequent_mask(T_dec)  # 形状 (T, T) = (10, 10)，dtype=float32
# .bool() 将 float 掩码转为 bool：-inf → True（忽略），0.0 → False（保留）
# 与 tgt_key_padding_mask（bool）类型统一，避免 PyTorch 弃用警告
tgt_mask = tgt_mask.bool()  # 形状 (T, T) = (10, 10)，dtype=bool；True=不可见，False=可见
print('因果掩码形状:', tgt_mask.shape)         # -> torch.Size([10, 10])
print('因果掩码（截取前4行4列）:\n', tgt_mask[:4, :4])  # 左下角 False（可见），右上角 True（不可见）

# ── 构造 padding 掩码 ──────────────────────────────────────────────────
# memory_key_padding_mask：与 Encoder 端的 src_key_padding_mask 保持一致
# 告知 Decoder 交叉注意力：Encoder 输出的哪些位置是 padding，不参与交叉注意力
memory_key_padding_mask = torch.zeros(N_dec, S_dec, dtype=torch.bool)  # 形状 (2, 15)，初始全 False
memory_key_padding_mask[0, 12:] = True  # 第 0 个样本：Encoder 侧位置 12~14 是 padding

# ── 构造目标序列 padding 掩码 ────────────────────────────────────────
# tgt_key_padding_mask：形状 (N, T)，dtype=bool
# True  = padding 位置，Decoder 自注意力时忽略该位置
# False = 正常 token，正常参与注意力计算
tgt_key_padding_mask = torch.zeros(N_dec, T_dec, dtype=torch.bool)  # 初始全 False，形状 (2, 10)
tgt_key_padding_mask[0, 8:] = True  # 第 0 个样本：目标位置 8、9 是 padding

print('memory 形状:', memory.shape)                                    # -> torch.Size([2, 15, 256])
print('tgt 形状:', tgt.shape)                                          # -> torch.Size([2, 10, 256])
print('tgt_key_padding_mask 形状:', tgt_key_padding_mask.shape)        # -> torch.Size([2, 10])，dtype=bool
print('memory_key_padding_mask 形状:', memory_key_padding_mask.shape)  # -> torch.Size([2, 15])

因果掩码形状: torch.Size([10, 10])
因果掩码（截取前4行4列）:
 tensor([[False,  True,  True,  True],
        [False, False,  True,  True],
        [False, False, False,  True],
        [False, False, False, False]])
memory 形状: torch.Size([2, 15, 256])
tgt 形状: torch.Size([2, 10, 256])
tgt_key_padding_mask 形状: torch.Size([2, 10])
memory_key_padding_mask 形状: torch.Size([2, 15])


In [36]:
# ── 执行 Decoder 前向传播 ──────────────────────────────────────────────
decoder.eval()  # 切换评估模式，关闭 Dropout

with torch.no_grad():  # 讲解演示，不需要梯度
    dec_out = decoder(
        tgt,                                               # 目标序列嵌入，形状 (N, T, E) = (2, 10, 256)
        memory,                                            # Encoder 输出，形状 (N, S, E) = (2, 15, 256)
        tgt_mask=tgt_mask,                                 # 因果掩码，形状 (T, T) = (10, 10)；-inf=不可见
        tgt_key_padding_mask=tgt_key_padding_mask,         # 目标序列 padding 掩码，形状 (N, T) = (2, 10)；True=忽略
        memory_key_padding_mask=memory_key_padding_mask,   # Encoder 输出 padding 掩码，形状 (N, S) = (2, 15)；True=忽略
    )  # 返回值：Tensor，形状 (N, T, E) = (2, 10, 256)

print('Decoder 输出形状:', dec_out.shape)
# -> torch.Size([2, 10, 256])
# 输出序列长度与 tgt 相同（T=10），每个目标位置融合了来自 Encoder 的上下文信息

Decoder 输出形状: torch.Size([2, 10, 256])


## 六、nn.Transformer（完整 Encoder-Decoder）使用讲解

`nn.Transformer` 是对 `nn.TransformerEncoder` + `nn.TransformerDecoder` 的一站式封装，适用于机器翻译等序列到序列任务。

- 内部已自动串联 Encoder 和 Decoder
- 同样**不含词嵌入和位置编码**，使用者须在外部完成

In [37]:
# ── 构造完整 Transformer ───────────────────────────────────────────────
transformer = nn.Transformer(
    d_model=512,             # 特征（嵌入）维度，int；Encoder 和 Decoder 共享此维度
    nhead=8,                 # 多头注意力头数，int；d_model // nhead = 64（每头维度）
    num_encoder_layers=6,    # Encoder 堆叠层数，int
    num_decoder_layers=6,    # Decoder 堆叠层数，int
    dim_feedforward=2048,    # FFN 隐藏层维度，int；论文原版为 4 × d_model
    dropout=0.1,             # Dropout 概率，float
    batch_first=True,        # bool；True 表示输入形状为 (N, S, E) 而非 (S, N, E)
)  # 返回值：nn.Transformer 实例，内部包含 encoder 和 decoder 两个子模块

# 打印参数量（单位：百万 M）
total_params = sum(p.numel() for p in transformer.parameters())  # 返回值：int，模型总参数数量
print(f'模型总参数量：{total_params / 1e6:.1f} M')  # 约 44 M，与论文 base 版本接近

模型总参数量：44.1 M


In [38]:
# ── 构造完整的输入数据与掩码 ──────────────────────────────────────────
N_full, S_full, T_full, E_full = 2, 10, 7, 512  # N=batch, S=源序列长, T=目标序列长, E=特征维度

# src：已完成词嵌入 + 位置编码的源序列特征（实际使用时须自行叠加 PositionalEncoding）
src_full = torch.rand(N_full, S_full, E_full)  # 形状 (2, 10, 512)，dtype=float32，模拟德语输入

# tgt：已完成词嵌入 + 位置编码的目标序列特征（训练时使用 teacher-forcing）
tgt_full = torch.rand(N_full, T_full, E_full)  # 形状 (2, 7, 512)，dtype=float32，模拟英语输入

# ── 掩码一：因果掩码（Decoder 自注意力用）────────────────────────────
# 形状 (T, T) = (7, 7)，不含 batch 和 num_heads 维度
# 生成 float 掩码后立即转 bool：-inf → True（忽略），0.0 → False（保留）
# 与 tgt_key_padding_mask_full（bool）类型统一，避免弃用警告
tgt_mask_full = nn.Transformer.generate_square_subsequent_mask(T_full).bool()  # 形状 (7, 7)，dtype=bool

# ── 掩码二：源序列 padding 掩码（Encoder 端）──────────────────────────
# 形状 (N, S) = (2, 10)，bool 类型；True 表示该位置是 padding
src_key_padding_mask_full = torch.zeros(N_full, S_full, dtype=torch.bool)  # 初始全 False
src_key_padding_mask_full[0, 8:] = True  # 第 0 个样本：位置 8、9 是 padding（该句较短）

# ── 掩码三：目标序列 padding 掩码（Decoder 端）────────────────────────
# 形状 (N, T) = (2, 7)，dtype=bool；True=padding 忽略，False=正常
tgt_key_padding_mask_full = torch.zeros(N_full, T_full, dtype=torch.bool)  # 初始全 False
tgt_key_padding_mask_full[0, 6:] = True  # 第 0 个样本：目标序列最后 1 个位置是 padding

print('src 形状:', src_full.shape)                                    # -> torch.Size([2, 10, 512])
print('tgt 形状:', tgt_full.shape)                                    # -> torch.Size([2, 7, 512])
print('tgt_mask 形状:', tgt_mask_full.shape)                          # -> torch.Size([7, 7])
print('src_key_padding_mask 形状:', src_key_padding_mask_full.shape)  # -> torch.Size([2, 10])
print('tgt_key_padding_mask 形状:', tgt_key_padding_mask_full.shape)  # -> torch.Size([2, 7])，dtype=bool

src 形状: torch.Size([2, 10, 512])
tgt 形状: torch.Size([2, 7, 512])
tgt_mask 形状: torch.Size([7, 7])
src_key_padding_mask 形状: torch.Size([2, 10])
tgt_key_padding_mask 形状: torch.Size([2, 7])


In [39]:
# ── 执行完整 Transformer 前向传播 ─────────────────────────────────────
transformer.eval()  # 切换评估模式，关闭 Dropout

with torch.no_grad():  # 讲解演示，不需要梯度
    out_full = transformer(
        src_full,                                           # 已嵌入的源序列，形状 (N, S, E) = (2, 10, 512)
        tgt_full,                                           # 已嵌入的目标序列，形状 (N, T, E) = (2, 7, 512)
        tgt_mask=tgt_mask_full,                             # Decoder 因果掩码，形状 (T, T) = (7, 7)
        src_key_padding_mask=src_key_padding_mask_full,     # Encoder padding 掩码，形状 (N, S) = (2, 10)
        tgt_key_padding_mask=tgt_key_padding_mask_full,     # Decoder 输入 padding 掩码，形状 (N, T) = (2, 7)
        memory_key_padding_mask=src_key_padding_mask_full,  # 交叉注意力 padding 掩码，与 src 保持一致
    )  # 返回值：Tensor，形状 (N, T, d_model) = (2, 7, 512)

print('Transformer 输出形状:', out_full.shape)
# -> torch.Size([2, 7, 512])
# 输出为 Decoder 每个目标位置的上下文特征向量
# 后续需再接 nn.Linear(d_model, vocab_size) 将其映射为词汇表上的概率分布

Transformer 输出形状: torch.Size([2, 7, 512])


## 七、接入线性分类头：从特征到词汇概率

Transformer 输出的 `(N, T, d_model)` 特征不能直接用于预测，需要接一个线性层映射到词汇表。

In [40]:
# ── 词汇表大小（假设英语目标端词表大小）────────────────────────────────
tgt_vocab_size = 8000  # 目标语言词汇表大小，int；实际项目中约 8k~32k

# ── 线性分类头：将 d_model 维特征映射到词汇表概率分布 ─────────────────
# 输入：(N, T, d_model)，输出：(N, T, tgt_vocab_size)
output_projection = nn.Linear(
    E_full,          # in_features = d_model = 512，int；与 Transformer 输出维度一致
    tgt_vocab_size,  # out_features = 词汇表大小，int
    bias=True,       # bool；是否添加偏置项，通常保留
)  # 返回值：nn.Linear 实例，参数量 = 512 × 8000 + 8000 = 4,104,000

# ── 计算 logits ──────────────────────────────────────────────────────
with torch.no_grad():  # 演示阶段不计算梯度
    logits = output_projection(out_full)  # 输入形状 (2, 7, 512)，输出形状 (2, 7, 8000)，dtype=float32
print('logits 形状:', logits.shape)  # -> torch.Size([2, 7, 8000])

# ── 计算概率分布（推理时）───────────────────────────────────────────
# dim=-1 表示对最后一维（词汇表维度）做 softmax，得到每个位置上各词的概率
probs = F.softmax(logits, dim=-1)  # 形状 (2, 7, 8000)，各位置概率和为 1
print('probs 形状:', probs.shape)   # -> torch.Size([2, 7, 8000])

# ── 贪心解码：取每个位置概率最高的词 token id ──────────────────────
# argmax 沿词汇表维度取最大值索引
pred_ids = probs.argmax(dim=-1)  # 形状 (N, T) = (2, 7)，dtype=int64；每个位置预测的 token id
print('预测 token id 形状:', pred_ids.shape)  # -> torch.Size([2, 7])
print('预测 token id 示例:\n', pred_ids)       # 每行为一个样本的 7 个预测 token

logits 形状: torch.Size([2, 7, 8000])
probs 形状: torch.Size([2, 7, 8000])
预测 token id 形状: torch.Size([2, 7])
预测 token id 示例:
 tensor([[ 399, 7226,  399,  399, 7226, 7226, 7226],
        [5473, 5473, 3915, 3915, 3915, 3915, 3915]])


## 八、掩码差异总结对照表

> **N** = batch size；**S** = Source 源序列长；**T** = Target 目标序列长；**nhead** = 注意力头数

| 属性 | Notebook 2 手写实现 | nn.Transformer |
|:---|:---|:---|
| **掩码张量形状** | `(batch, 1, seq_q, seq_k)` 合并 padding + causal | causal: `(T, T)` / padding: `(N, S)` 或 `(N, T)` 两者分开传入 |
| **batch 维度** | 有（dim=0），每个样本独立控制 | causal 掩码无 batch 维，内部自动广播至所有样本 |
| **num_heads 维度** | 固定为 1（dim=1），广播至所有头 | 完全没有，由框架内部扩展 |
| **bool True 的语义** | True = 被遮盖（乘以 −1e9，softmax 趋近 0） | `_mask` / `_key_padding_mask`：True = 忽略该位置 |
| **与 F.sdpa bool 对比** | N/A（使用自定义乘法掩码） | ⚡ F.sdpa True = 保留；nn.Transformer True = 忽略，**语义相反！** |
| **因果掩码值类型** | 整数/浮点掩码直接 `× (−1e9)` 加到注意力分数 | `generate_square_subsequent_mask` 生成后 `.bool()` 转换 |
| **padding 掩码构造** | 在 DataLoader 中构造，合并到 `attn_mask` 张量 | 直接传 `(N, S)` / `(N, T)` bool 张量，框架内部处理广播与合并 |

> **为什么 nn.Transformer 与 F.sdpa 的 bool 语义相反？**
>
> `nn.Transformer` 内部最终会调用 `F.scaled_dot_product_attention`（SDPA）来执行实际的注意力计算。
> 但两者对 bool 掩码的约定不同：
> - `nn.Transformer` 对外暴露的接口：`True = 忽略该位置`（对用户友好的语义）
> - `F.scaled_dot_product_attention` 的 `attn_mask`（bool）：`True = 保留该位置`
>
> PyTorch 在内部会自动做一次取反（`~mask`）再传给 SDPA，因此两边语义看似相反，
> 实际上 `nn.Transformer` 帮用户屏蔽了这一细节。
> 只有在**直接调用 `F.scaled_dot_product_attention`** 时才需要注意这个区别。

## 九、关键结论

1. **形状差异根本原因**：`nn.Transformer` 将掩码职责拆分（因果 vs padding），由框架内部完成维度扩展，无需用户手动管理 `num_heads` 维；手写实现将两者合并在 `(batch, 1, seq_q, seq_k)` 张量中，利用广播机制节省内存。

2. **bool 语义要分清楚**：
   - `nn.Transformer` 的掩码参数：`True = 忽略该位置`
   - `F.scaled_dot_product_attention` 的 `attn_mask`（bool）：`True = 保留该位置`
   - 混用会导致掩码效果完全相反，是常见 Bug 来源

3. **实用建议**：快速实验或微调场景优先使用 `nn.Transformer` 系列接口（代码量少、掩码职责清晰）；需要高度定制化（如 RoPE、GEGLU、Pre-LN 等）时再手写各模块。